# 16 - Management Consulting Cost Optimizer
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/16-mgmt-consulting/mgmt_consulting_workbook.ipynb)

Analyse an operational profile and place each identified inefficiency into a 2x2 effort-impact matrix -- quick wins first.

**Harness focus:** Structured diagnosis + 2D effort-impact ranking

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: The 2x2 effort-impact matrix

The classic management consulting prioritization tool:

```
           LOW EFFORT          HIGH EFFORT
          +-----------------+------------------+
HIGH      | QUICK WIN       | MAJOR PROJECT    |
IMPACT    | Do first        | Plan carefully   |
          +-----------------+------------------+
LOW       | FILL-IN         | THANKLESS TASK   |
IMPACT    | When bandwidth  | Avoid or skip    |
          +-----------------+------------------+
```

The schema enforces this: `quick_wins`, `major_projects`, `fill_ins`, and `thankless_tasks` are separate typed lists. The model cannot mix them up.

## Part 2: Schema -- quadrant as a typed Literal field

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel

class Recommendation(BaseModel):
    title: str
    category: Literal["process", "technology", "people", "structure"]
    effort: Literal["low", "medium", "high"]
    impact: Literal["low", "medium", "high"]
    quadrant: Literal["quick_win", "major_project", "fill_in", "thankless_task"]
    estimated_annual_saving: Optional[str] = None
    rationale: str
    implementation_steps: List[str]

class CostOptimizationReport(BaseModel):
    company: Optional[str] = None
    total_addressable_saving: Optional[str] = None
    executive_summary: str
    quick_wins: List[Recommendation]
    major_projects: List[Recommendation]
    fill_ins: List[Recommendation]
    thankless_tasks: List[Recommendation]
    prioritization_note: str

print("Schema defined.")

## Part 3: System prompt -- encode the quadrant logic

The system prompt must encode the quadrant assignment rules exactly. Without this, the model will inconsistently classify effort and impact.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

CONSULTANT_SYSTEM = SystemMessage("""
You are a management consultant conducting a cost optimization review.

For each inefficiency identified:
1. Classify effort to fix: low / medium / high
2. Classify savings potential (impact): low / medium / high
3. Assign quadrant using EXACTLY this logic:
   - effort=low  + impact=high --> quick_win
   - effort=high + impact=high --> major_project
   - effort=low  + impact=low  --> fill_in
   - effort=high + impact=low  --> thankless_task
   Treat medium effort as low; treat medium impact as high.
4. Estimate annual saving in GBP where data supports it.
5. List 3-5 concrete implementation steps.

Sort quick_wins first. Provide executive_summary and prioritization_note.
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
consultant = CONSULTANT_SYSTEM | llm.with_structured_output(CostOptimizationReport)
print("Agent ready.")

## Part 4: Run on the Meridian Logistics profile

In [ ]:
PROFILE = """
Meridian Logistics Ltd -- 280 employees, GBP 42m revenue.

Issues:
1. Manual invoice processing: 12 FTE, 3-day cycle, 8% error rate, GBP 720k cost
2. SaaS sprawl: 14 subscriptions, 40% overlap, GBP 310k/year
3. Excel shift scheduling: 22% overtime premium, GBP 890k overtime cost
4. Legacy ERP middleware: GBP 35-80k per integration, 3-6 month lead time
5. Three CRM systems: GBP 400k estimated lost cross-sell revenue
6. Manual HR onboarding: 2 weeks per hire, 45 hires/year
7. Print-based approvals: 4-7 day cycle, GBP 18k print/postage cost
"""

report = consultant.invoke(HumanMessage(content="Operational profile:\n\n" + PROFILE))
print(f"Quick wins: {len(report.quick_wins)} | Major projects: {len(report.major_projects)}")
print(f"Total saving: {report.total_addressable_saving}")

## Part 5: Print the matrix view -- quick wins first

In [ ]:
print(report.executive_summary)
print()

def show_quadrant(label, recs):
    if not recs:
        return
    print(f"{label} ({len(recs)}):")
    for r in recs:
        saving = f" | {r.estimated_annual_saving}" if r.estimated_annual_saving else ''
        print(f"  [{r.effort}/{r.impact}]{saving} {r.title}")
    print()

show_quadrant("QUICK WINS", report.quick_wins)
show_quadrant("MAJOR PROJECTS", report.major_projects)
show_quadrant("FILL-INS", report.fill_ins)
show_quadrant("THANKLESS TASKS", report.thankless_tasks)
print(f"Priority: {report.prioritization_note}")

## Exercise: Add implementation_risk and re-sort quick wins

Add `implementation_risk: Literal['low', 'medium', 'high']` to `Recommendation`. Re-sort the quick_wins list so low-risk wins appear first.

**Starter:** Add the field below.

In [ ]:
# Starter: add implementation_risk
class RecommendationV2(BaseModel):
    title: str
    category: Literal["process", "technology", "people", "structure"]
    effort: Literal["low", "medium", "high"]
    impact: Literal["low", "medium", "high"]
    quadrant: Literal["quick_win", "major_project", "fill_in", "thankless_task"]
    estimated_annual_saving: Optional[str] = None
    rationale: str
    implementation_steps: List[str]
    implementation_risk: Literal["low", "medium", "high"]  # NEW

class CostOptimizationReportV2(BaseModel):
    company: Optional[str] = None
    total_addressable_saving: Optional[str] = None
    executive_summary: str
    quick_wins: List[RecommendationV2]
    major_projects: List[RecommendationV2]
    fill_ins: List[RecommendationV2]
    thankless_tasks: List[RecommendationV2]
    prioritization_note: str

# TODO: update CONSULTANT_SYSTEM to assess implementation_risk
# TODO: rebuild and run, then sort quick_wins by implementation_risk ascending

In [ ]:
# Answer key
RISK_ORDER = {"low": 0, "medium": 1, "high": 2}

CONSULTANT_SYSTEM_V2 = SystemMessage("""
You are a management consultant conducting a cost optimization review.

For each inefficiency:
1-5. Apply quadrant logic and estimate savings as before.
6. Rate implementation_risk:
   low    = routine change, single team
   medium = cross-team coordination needed
   high   = significant change management or technology risk
""")

consultant_v2 = CONSULTANT_SYSTEM_V2 | llm.with_structured_output(CostOptimizationReportV2)
report_v2 = consultant_v2.invoke(HumanMessage(content="Operational profile:\n\n" + PROFILE))

sorted_qw = sorted(report_v2.quick_wins, key=lambda r: RISK_ORDER[r.implementation_risk])
print("Quick wins sorted by risk (lowest first):")
for r in sorted_qw:
    print(f"  [risk={r.implementation_risk}] {r.title}")

## What You Built

- A cost optimization consultant that places each recommendation in a 2x2 matrix
- `quadrant` as a typed Literal field that enforces correct classification
- Schema with separate typed lists per quadrant
- System prompt encoding the exact quadrant assignment rules

**Next steps:** Try example 17 (transaction readiness gating) or adapt this to a product roadmap prioritization use case.